# 01e — SSIC classification from the business name

Loads `mc_matched.parquet` (from `01c_id_matching`) and `labels.parquet` (from `01d_survival_label`), finds the SSIC
code of every business from its name, and saves `ssic_by_business.csv`, `business_names.csv` and `business_city.csv`
in the data folder for `03_official_activity`, `04_gastat_preprocessing` and `05_sama_preprocessing`.

**Rules for this notebook**
- The raw files are never changed. Anything cleaned is saved as a new file.
- Nothing is dropped. Problem rows get a flag column instead.
- Data never goes into Git. Clear all outputs before committing.

**The preprocessing notebooks, in order** (each one loads what the one before saved):
1. `01a_id_check`: did the ID change, or only its column name? (stands alone)
2. `01b_all_quarters`: reads the 30 Excel files (2023 Q1 – 2026 Q3) once and saves `mc_all_quarters.parquet`
3. `01c_id_matching`: one ID per business, saves `mc_matched.parquet`
4. `01d_survival_label`: the one-year label, saves `labels.parquet`
5. `01e_ssic_classification`: SSIC from the name, saves `ssic_by_business.csv`, `business_names.csv`, `business_city.csv`
6. `01f_old_data_branches`: old data (2023 Q1 – 2025 Q2), every branch joined to its main (stands alone)

### Cell 1 — Setup
Finds the data. On Colab it downloads the raw files from the team's Google Drive folder (temporary copies,
gone when the session ends). On a laptop it reads the local copy in `DATA_DIR`.

The results that the next notebook loads go in `SAVED_DIR`: the folder **saved** inside the team's shared Drive
folder **Darb DataSets**. Everyone loads the same results, and they are still there in a new Colab session.
On Colab: add a shortcut to Darb DataSets in your My Drive first (Drive → right-click the folder → Organize →
Add shortcut → My Drive). Colab asks once to connect Drive.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    !pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
    from google.colab import drive
    drive.mount('/content/drive')
    TEAM_DIR = Path('/content/drive/MyDrive/Darb DataSets')     # the team's shared folder (a shortcut in My Drive)
    if not TEAM_DIR.exists():
        raise FileNotFoundError('Darb DataSets is not in your My Drive: add a shortcut to it (see the text above), then run this cell again.')
    SAVED_DIR = TEAM_DIR / 'saved'                              # shared with the team, kept between Colab sessions
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', r'C:\Users\Nasser Alyahya\darb-data'))
    SAVED_DIR = DATA_DIR / 'saved'

MC_DIR = DATA_DIR / 'MC'
SAVED_DIR.mkdir(parents=True, exist_ok=True)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR, '| saved results:', SAVED_DIR)

### Load — the matched table and the label table
Only the columns this part needs from `mc`, to save time and memory.

In [ ]:
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PATH = SAVED_DIR / 'mc_matched.parquet'
if not PATH.exists():
    raise FileNotFoundError(f'{PATH} not found: run 01c_id_matching first.')
# Text is loaded in the same compact format as in 01c_id_matching (pd.read_parquet would use about twice the memory)
TEXT = pd.StringDtype('pyarrow')
mc = pq.read_table(PATH, columns=['entity_id', 'quarter', 'name', 'city', 'record_type', 'link_flag']).to_pandas(types_mapper={pa.string(): TEXT, pa.large_string(): TEXT}.get)
print(f'Loaded {PATH.name}: {len(mc):,} rows | memory used: {mc.memory_usage(deep=True).sum() / 1e9:.1f} GB')

In [ ]:
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PATH = SAVED_DIR / 'labels.parquet'
if not PATH.exists():
    raise FileNotFoundError(f'{PATH} not found: run 01d_survival_label first.')
# Text is loaded in the same compact format as in 01d_survival_label (pd.read_parquet would use about twice the memory)
TEXT = pd.StringDtype('pyarrow')
labels = pq.read_table(PATH).to_pandas(types_mapper={pa.string(): TEXT, pa.large_string(): TEXT}.get)
print(f'Loaded {PATH.name}: {len(labels):,} rows | memory used: {labels.memory_usage(deep=True).sum() / 1e9:.1f} GB')

In [ ]:
QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']

### Cell 16 — The SSIC divisions and a 300-name test sample
Loads the official SSIC file (Gastat) and lists its divisions. Then picks 300 random businesses from the
cohorts we train on (2024 Q4, 2025 Q1, 2025 Q2), one name each, and saves them to `ssic_test_sample.csv`
(a new file on the runtime's disk). A division is suggested for each name, and a person checks every one:
the checked divisions are the answer key used to compare the classification methods.

Needs Cell 1 and the two Load cells run first.

In [ ]:
SSIC_FILE = 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'
SSIC_PATH = DATA_DIR / 'Gastat' / SSIC_FILE
if IN_COLAB and not SSIC_PATH.exists():
    import gdown
    SSIC_PATH.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id='1wlS-84Ojsiv0UP8sjmZmNcENU_M8J0wZ', output=str(SSIC_PATH), quiet=True)

# The SSIC sheet starts with a title; the real header is the row that starts with "رمز الباب"
raw = pd.read_excel(SSIC_PATH, engine='calamine', header=None, dtype=str)
header_row = raw.index[raw[0].fillna('').str.strip() == 'رمز الباب'][0]
ssic = raw.iloc[header_row + 1:].copy()
ssic.columns = ['section_code', 'section_name', 'division_code', 'division_name', 'group_code', 'group_name',
                'class_code', 'class_name', 'activity_code', 'activity_name', 'activity_description',
                'supervising_authority', 'licensing_authority']
ssic = ssic.dropna(subset=['activity_code']).apply(lambda column: column.str.strip())

divisions = (ssic.groupby(['section_code', 'section_name', 'division_code', 'division_name'], sort=False)
             .size().rename('activities').reset_index())
print(f'SSIC: {len(ssic):,} activities | {ssic["section_code"].nunique()} sections | {len(divisions)} divisions')
print()

# 300 random businesses from the cohorts we will train on (one name per business: its latest name)
pool = pd.Series(labels.index[labels['cohort_quality'].isin(['clean', 'start_incomplete'])])
sample_ids = pool.sample(300, random_state=42)
rows = mc.loc[mc['entity_id'].isin(sample_ids) & (mc['record_type'] == 'رئيسي'), ['entity_id', 'quarter', 'name', 'city']]
rows = rows.assign(order=rows['quarter'].map({q: i for i, q in enumerate(QUARTERS)}).astype(int))
sample = (rows.sort_values('order').drop_duplicates('entity_id', keep='last')
          [['entity_id', 'name', 'city']].sort_values('entity_id').reset_index(drop=True))
sample.index = sample.index + 1
sample['suggested_division'] = ''     # filled in next, then checked by a person
sample['correct_division'] = ''       # the answer key: the division a person confirms

sample.to_csv(DATA_DIR / 'ssic_test_sample.csv', index_label='no', encoding='utf-8-sig')
print(f'Saved {len(sample)} names to {DATA_DIR / "ssic_test_sample.csv"}')
print()
for no, row in sample.iterrows():
    print(f'{no}|{row["entity_id"]}|{row["name"]}|{row["city"]}')


### Cell 17 — Suggest a division for the 300 test names
The MC files have **no activity or SSIC column**, so the SSIC code is found from the business name with the
rules in `ml/src/isic_classifier.py`. The classifier first checks itself on a few known names, then fills
`ssic_section`, `suggested_division`, `ssic_flag` and `ssic_candidates` for the 300 names and saves
`ssic_test_sample_suggested.csv`.

**Then a person checks every name** (open the file in Excel or Google Sheets):
- Write the right 2-digit SSIC division in `correct_division` (e.g. `56` for restaurants, `47` for retail).
  Look the code up in the SSIC file, not in the suggestion.
- If the name alone does not tell the activity (e.g. "مؤسسة سالم محمد العتيبي"), write `?`.
- Do not change the other columns. Save as `ssic_test_sample_checked.csv` in the data folder
  (on Colab: upload it to `/content/darb-data`).

Needs Cell 16 run first.

In [ ]:
import sys
import importlib

# Where isic_classifier.py is: next to the notebook on Colab (upload it), ml/src on a laptop
SRC_DIR = Path('/content') if IN_COLAB else Path.cwd().parent / 'src'
if not (SRC_DIR / 'isic_classifier.py').exists():
    raise FileNotFoundError(f'Put isic_classifier.py in {SRC_DIR} first (on Colab: Files panel -> Upload).')
sys.path.insert(0, str(SRC_DIR))
import isic_classifier
importlib.reload(isic_classifier)                 # picks up rule changes without a restart
from isic_classifier import classify_tuple, RESULT, self_test

# The classifier must pass its own examples before it is used
assert self_test(), 'The classifier failed its self-test: fix isic_classifier.py first.'

# The SSIC file writes sections with Arabic letters (أ، ب، ج …), the classifier with ISIC letters (A, B, C …).
# Both follow the same order, so the n-th section of the file is the n-th letter.
arabic_sections = list(dict.fromkeys(divisions['section_code']))
SECTION_LETTER = dict(zip(arabic_sections, 'ABCDEFGHIJKLMNOPQRSTU'))
divisions['section_letter'] = divisions['section_code'].map(SECTION_LETTER)
letter_of_division = dict(zip(divisions['division_code'].str.zfill(2), divisions['section_letter']))
assert [letter_of_division[d] for d in ['01', '10', '41', '47', '56', '86']] == ['A', 'C', 'F', 'G', 'I', 'Q'], \
    'The Arabic section letters do not line up with A, B, C …: check the SSIC file.'
print('SSIC sections:', ', '.join(f'{a} = {l}' for a, l in SECTION_LETTER.items()))

# Suggest a division for every name of the test sample
suggested = pd.DataFrame([classify_tuple(n) for n in sample['name']], index=sample.index, columns=RESULT)
sample['ssic_section'] = suggested['ssic_section']
sample['suggested_division'] = suggested['ssic_division']
sample['ssic_flag'] = suggested['ssic_flag']
sample['ssic_candidates'] = suggested['ssic_candidates']
sample = sample[['entity_id', 'name', 'city', 'ssic_section', 'suggested_division', 'ssic_flag',
                 'ssic_candidates', 'correct_division']]

sample.to_csv(DATA_DIR / 'ssic_test_sample_suggested.csv', index_label='no', encoding='utf-8-sig')
print(f'Saved {len(sample)} names to {DATA_DIR / "ssic_test_sample_suggested.csv"}')
print()
print('Flags in the sample:')
print(sample['ssic_flag'].value_counts().to_string())


### Cell 18 — How accurate is the classifier? (only counts)
Reads the checked sample `ssic_test_sample_checked.csv` and compares the classifier with the answer key:
- **Coverage:** share of names that got a section (not flagged `ambiguous` / `unclassified`).
- **Section accuracy:** of those, how many have the right section.
- **Division accuracy:** of the names flagged `ok`, how many have the right division.
- **Missed:** names the classifier flagged, but a person could classify.

The wrong ones are listed, to improve the rules in `isic_classifier.py`. After changing a rule, run Cells 17 and 18
again (the answer key stays the same). Write the final numbers in the notes at the end of Part 5.

In [ ]:
CHECKED = DATA_DIR / 'ssic_test_sample_checked.csv'
# sep=None: works for a comma file (saved from Excel) and a tab file (pasted with %%writefile)
checked = pd.read_csv(CHECKED, dtype=str, encoding='utf-8-sig', sep=None, engine='python').fillna('')
for column in ['ssic_section', 'suggested_division', 'correct_division']:
    checked[column] = checked[column].str.strip()

# The answer key: a 2-digit SSIC division, or "?" when a person cannot tell from the name either
is_code = checked['correct_division'].str.fullmatch(r'\d{1,2}')
checked.loc[is_code, 'correct_division'] = checked.loc[is_code, 'correct_division'].str.zfill(2)
section_of = letter_of_division                            # division -> section letter (Cell 17)
empty = checked['correct_division'] == ''
cannot_tell = checked['correct_division'] == '?'
not_a_code = ~empty & ~cannot_tell & ~checked['correct_division'].isin(section_of)
print(f'Checked file: {len(checked)} names | not filled in: {empty.sum()} | "?": {cannot_tell.sum()} | '
      f'not an SSIC division: {not_a_code.sum()}')
if empty.any() or not_a_code.any():
    print('Fix these rows first:')
    print(checked.loc[empty | not_a_code, ['no', 'name', 'correct_division']].to_string(index=False))

checked['correct_section'] = checked['correct_division'].map(section_of).fillna('')
judged = checked[checked['correct_section'] != '']          # names a person could classify
given = judged['ssic_section'] != ''                        # names the classifier gave a section

section_right = judged['ssic_section'] == judged['correct_section']
division_rows = judged['ssic_flag'] == 'ok'
division_right = judged['suggested_division'] == judged['correct_division']

print()
print(f'Names a person could classify: {len(judged)} of {len(checked)}')
print(f'Coverage   (classifier gave a section):          {given.mean():.1%}')
print(f'Section accuracy  (where a section was given):   {section_right[given].mean():.1%}')
print(f'Division accuracy (where the flag is "ok"):      {division_right[division_rows].mean():.1%}')
print(f'Missed (flagged, but a person could tell):       {(~given).sum()}')

print()
print('By flag:')
by_flag = judged.assign(section_right=section_right, division_right=division_right).groupby('ssic_flag').agg(
    names=('name', 'size'), section_right=('section_right', 'mean'), division_right=('division_right', 'mean'))
print(by_flag.to_string(float_format='{:.1%}'.format))

print()
print('Wrong sections (to fix the rules):')
wrong = judged[given & ~section_right]
print(wrong[['no', 'name', 'ssic_section', 'suggested_division', 'ssic_candidates', 'correct_division']].to_string(index=False))


### Cell 19 — Classify every business (adds columns, deletes nothing)
1. **One name per business:** the name of its main record in its **first quarter** (when it started). That is what
   is known on day one; a later name can be clearer only because the business survived long enough to rename
   itself, which would leak the outcome. (For the 2025 Q3 cohort, whose start file is missing, this is the first
   quarter it appears in.) The code of the **latest** name is kept next to it (`ssic_latest_*`) for comparison
   only, never for the model.
2. Each name gets a section, a division and an `ssic_flag` from the classifier.
3. **Consistency:** a business gets one code for all quarters. If its names in different quarters point to
   different sections, the latest name's code is kept but the flag becomes `names_disagree`.
4. **Check:** every code given must exist in the official SSIC file (Cell 16).
5. The columns are added to `labels` (one row per business) and saved to `ssic_by_business.csv`.
6. The start and latest name of each business are saved to `business_names.csv` (data folder only, never in Git),
   for the keyword work in `03_official_activity`.
7. The city of each business in its **first quarter** (main record) is saved to `business_city.csv` (data folder
   only), for the SAMA city features in `05_sama_preprocessing`.

`ssic_flag`: `ok`, `division_unclear`, `low_confidence`, `ambiguous`, `unclassified`, `names_disagree`.
Only `ok` and `division_unclear` have a sure section. The other flags are kept as they are, never filled in by guessing.

Needs Cell 1, the Load cells, 16 and 17 run first.

In [ ]:
from isic_classifier import classify_tuple, RESULT

# 1. One name per business: its latest name (main records only, like the label table and the test sample)
QUARTER_ORDER = {q: i for i, q in enumerate(QUARTERS)}
use = mc.loc[(mc['record_type'] == 'رئيسي')
             & mc['link_flag'].isin(['same_id', 'linked_new_id', 'ambiguous_not_linked'])
             & mc['name'].notna(), ['entity_id', 'quarter', 'name']]
use = use.assign(order=use['quarter'].map(QUARTER_ORDER).astype(int))
names = use.drop_duplicates(['entity_id', 'name'])             # every different name a business had
del use
gc.collect()

# 2. Classify each different name once (many businesses share a name pattern; this saves time)
result_of = {name: classify_tuple(name) for name in names['name'].unique()}
print(f'Different names classified: {len(result_of):,}')

# The model uses the name the business had when it STARTED (its first quarter): what is known on day one.
# A later name can be clearer only because the business survived long enough to rename itself (a small leak).
by_order = names.sort_values('order')
first = by_order.drop_duplicates('entity_id', keep='first').set_index('entity_id')['name']
latest = by_order.drop_duplicates('entity_id', keep='last').set_index('entity_id')['name']
del by_order
ssic_by_business = pd.DataFrame([result_of[n] for n in first], index=first.index, columns=RESULT)

# The latest name's code is kept for comparison only (03_official_activity compares it with the official file,
# which is from the end of 2025). It is NOT for the model.
LATEST_COLUMNS = ['ssic_latest_section', 'ssic_latest_division', 'ssic_latest_label', 'ssic_latest_flag']
ssic_latest = pd.DataFrame([[result_of[n][i] for i in (0, 1, 2, 4)] for n in latest],
                           index=latest.index, columns=LATEST_COLUMNS).reindex(ssic_by_business.index)
renamed = first != latest.reindex(first.index)
changed = ssic_by_business['ssic_section'] != ssic_latest['ssic_latest_section']
print(f'Businesses whose name changed: {renamed.mean():.1%} | whose section changed with it: {changed.mean():.1%}')

# 3. Consistency: one code per business in every quarter. If its names point to different sections,
#    the latest name's code is kept but the business is flagged "names_disagree".
section_per_name = names.assign(section=names['name'].map(lambda n: result_of[n][0]))
section_per_name = section_per_name[section_per_name['section'] != '']
sections_per_business = section_per_name.groupby('entity_id')['section'].agg(lambda s: '|'.join(sorted(set(s))))
disagree = sections_per_business[sections_per_business.str.contains('|', regex=False)]
ssic_by_business.loc[disagree.index, 'ssic_flag'] = 'names_disagree'
ssic_by_business.loc[disagree.index, 'ssic_candidates'] = disagree
del names, section_per_name, sections_per_business, result_of
gc.collect()

# 4. Every code given must exist in the official SSIC file (section letters from Cell 17)
official = set(zip(divisions['section_letter'], divisions['division_code'].str.zfill(2)))
given = ssic_by_business[ssic_by_business['ssic_division'] != '']
not_official = given[[pair not in official for pair in zip(given['ssic_section'], given['ssic_division'])]]
print(f'Codes not found in the SSIC file: {len(not_official):,}', '(fix the rule)' if len(not_official) else '')
if len(not_official):
    print(not_official[['ssic_section', 'ssic_division', 'ssic_label']].value_counts().to_string())

# The official Arabic names of the section and division
section_names = dict(zip(divisions['section_letter'], divisions['section_name']))
division_names = dict(zip(divisions['division_code'].str.zfill(2), divisions['division_name']))
ssic_by_business['ssic_section_name'] = ssic_by_business['ssic_section'].map(section_names).fillna('')
ssic_by_business['ssic_division_name'] = ssic_by_business['ssic_division'].map(division_names).fillna('')

# 5. Add to the label table (one row per business). A business with no name at all stays unclassified.
SSIC_COLUMNS = RESULT + ['ssic_section_name', 'ssic_division_name']
labels = (labels.drop(columns=SSIC_COLUMNS + LATEST_COLUMNS, errors='ignore')
          .join(ssic_by_business[SSIC_COLUMNS]).join(ssic_latest))
labels['ssic_flag'] = labels['ssic_flag'].fillna('unclassified')
labels[SSIC_COLUMNS] = labels[SSIC_COLUMNS].fillna('')
labels[LATEST_COLUMNS] = labels[LATEST_COLUMNS].fillna('')
labels['ssic_latest_flag'] = labels['ssic_latest_flag'].replace('', 'unclassified')
labels['ssic_flag'] = labels['ssic_flag'].astype('category')
del ssic_by_business, ssic_latest, renamed, changed, given, not_official
gc.collect()

# 6. Counts
labelled = labels[labels['cohort_quality'] != 'cannot_judge']
flags = pd.DataFrame({'all businesses': labels['ssic_flag'].value_counts(),
                      'labelled cohorts': labelled['ssic_flag'].value_counts()})
flags['% of labelled'] = (flags['labelled cohorts'] / len(labelled)).map('{:.1%}'.format)
print()
print('ssic_flag (nothing is guessed: every business without a sure code is flagged):')
print(flags.to_string())
print()
print('Labelled businesses per section (flag ok or division_unclear):')
sure = labelled[labelled['ssic_flag'].isin(['ok', 'division_unclear'])]
print(sure.groupby(['ssic_section', 'ssic_section_name']).size().sort_values(ascending=False).to_string())

# 7. Save (a new file on the runtime's disk, never in Git)
labels[['cohort', 'cohort_quality', 'survived_1y'] + SSIC_COLUMNS + LATEST_COLUMNS].to_csv(
    DATA_DIR / 'ssic_by_business.csv', index_label='entity_id', encoding='utf-8-sig')
print()
print(f'Saved {len(labels):,} businesses to {DATA_DIR / "ssic_by_business.csv"}')

# The start and latest name of each business, for the keyword work in 03_official_activity (private: data folder only)
pd.DataFrame({'start_name': first, 'latest_name': latest.reindex(first.index)}).to_csv(
    DATA_DIR / 'business_names.csv', index_label='entity_id', encoding='utf-8-sig')
print(f'Saved {len(first):,} business names (start and latest) to {DATA_DIR / "business_names.csv"}')
del first, latest
# The city of each business in its FIRST quarter (main record), for the SAMA city features in 05_sama_preprocessing
city_rows = mc.loc[(mc['record_type'] == 'رئيسي')
                   & mc['link_flag'].isin(['same_id', 'linked_new_id', 'ambiguous_not_linked'])
                   & mc['city'].notna(), ['entity_id', 'quarter', 'city']]
first_city = (city_rows.assign(order=city_rows['quarter'].map(QUARTER_ORDER).astype(int))
              .sort_values('order').drop_duplicates('entity_id').set_index('entity_id')['city'].astype(str))
first_city.rename('city').to_csv(DATA_DIR / 'business_city.csv', index_label='entity_id', encoding='utf-8-sig')
print(f'Saved the first-quarter city of {len(first_city):,} businesses to {DATA_DIR / "business_city.csv"}')
del city_rows, first_city
gc.collect()


### Notes — SSIC mapping (fill in after running Cells 17–19)
- **Method:** the SSIC code is found from the business name (keyword rules, `ml/src/isic_classifier.py`),
  because the MC files have no activity column.
- **Test sample (300 names):** coverage __%, section accuracy __%, division accuracy __%. Checked by: __.
- **All labelled businesses:** `ok` __%, `division_unclear` __%, `low_confidence` __%, `ambiguous` __%,
  `unclassified` __%, `names_disagree` __%.
- **Known limits:** names that do not say the activity (personal names) cannot be classified; names with
  no Arabic letters are `unclassified`; a business is classified by its latest name.